In [1]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn #neural networks
import torch.nn.functional as F # activation functions
import torch.optim as optim #optimisers
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, root_mean_squared_log_error
from torch.utils.data import Dataset, DataLoader, TensorDataset, Subset

from sklearn.compose import ColumnTransformer

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder



In [2]:
train = pd.read_csv('data/train.csv')

print(f'Training set: {train.shape[0]} rows, {train.shape[1]} columns')
train.head()

Training set: 1460 rows, 81 columns


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## Defining the NN

In [3]:
class MyNn(nn.Module):
    def __init__(self, shape):
        super(MyNn, self).__init__()
        self.fc1 = nn.Linear(in_features=shape, out_features=64) # 79 [ID and SalePrice dropped
        self.fc2 = nn.Linear(in_features=64, out_features=50)
        self.fc4 = nn.Linear(in_features=50, out_features=35)
        self.fc3 = nn.Linear(in_features=35, out_features=1)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.leaky_relu(self.fc4(x))
        x = self.dropout(x)
        x = self.fc3(x)
        return x

## Preprocessing data
-> Filling NA with 0 if numerical, with a string ("Missing") if categorical
-> Grouping the columns in numerical and categorical for preprocessing

In [ ]:
# Separating features from target
X = train.drop(["SalePrice", "Id"], axis=1)
y_original = train["SalePrice"].copy()
y = np.log(train["SalePrice"])


num_col = X.select_dtypes(include=np.number).columns.tolist()
cat_col = X.select_dtypes(include=["object", "str"]).columns.tolist()
for col in num_col:
    X[col] = X[col].fillna(0)

for col in cat_col:
    X[col] = X[col].fillna("Missing").astype(str)

## Training

In [6]:

fold_mae_results = []
fold_mape_results = []
fold_rmse_results = []



# Splitting data into folds for cross-validation 
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(kfold.split(X)):
    print(f"----FOLD {fold+1}---")
    X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
    y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]

    # PREPROCESSOR: Applies the StandardScaler to the numerical features and the OneHotEncoder to the categorical ones
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), num_col),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_col)
        ]
    )

    # Standardizing 
    scaler_y = StandardScaler()
    X_train_transformed = preprocessor.fit_transform(X_train_fold)
    y_train_transformed = scaler_y.fit_transform(np.array(y_train_fold).reshape(-1, 1))


    X_val_trans = preprocessor.transform(X_val_fold)
    y_val_trans = scaler_y.transform(np.array(y_val_fold).reshape(-1, 1))
    
    price_mean = scaler_y.mean_[0]
    price_std = scaler_y.scale_[0]

    X_train_tensor = torch.tensor(X_train_transformed, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train_transformed, dtype=torch.float32)
    
    X_val_tensor = torch.tensor(X_val_trans, dtype=torch.float32)
    y_val_tensor = torch.tensor(y_val_trans, dtype=torch.float32)
    
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(train_dataset, batch_size=35, shuffle=True)
    
    val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
    val_loader = DataLoader(val_dataset, batch_size=35, shuffle=False)

    # ------- Model setup
    shape = X_train_tensor.shape[1]
    model = MyNn(shape)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0005)

    #------.

    # -------------------------------------------------------------------------------- TRAINING 
    model.train()
    epochs = 100

    for epoch in range(epochs):
        running_loss = 0.0
        for x_batch, y_batch in train_loader:
            optimizer.zero_grad()
            predictions = model.forward(x_batch)
            loss = criterion(predictions, y_batch)

            optimizer.zero_grad()
            loss.backward()
            total_norm = 0
            for p in model.parameters():
                if p.grad is not None:
                    total_norm += p.grad.detach().norm(2)
            optimizer.step()
            running_loss += loss.item()
        if (epoch + 1) == 1 or (epoch + 1) == 100:
            print(f"Epoch {epoch + 1} | Loss: {running_loss / len(train)}")

    # -------------------------------------------------------------------------------- MODEL EVALUATION            
    model.eval()

    with torch.no_grad():
        predictions = model(X_val_tensor)
        predictions_descaled = np.exp(predictions.numpy() * price_std + price_mean)

        y_val_real = np.exp(y_val_tensor.numpy() * price_std + price_mean)


        mae = mean_absolute_error(y_val_real,predictions_descaled)
        mape = mean_absolute_percentage_error(y_val_real,predictions_descaled)*100
        RMSE = root_mean_squared_log_error(y_val_real,predictions_descaled)

        fold_mae_results.append(mae)
        fold_mape_results.append(mape)
        fold_rmse_results.append(RMSE)


print(f"\nOverall performance:")
print(f"RMSE: {RMSE:.2f}")
print(f"  Mean Absolute Error:  £{mae:,.0f}")
print(f"  Mean Absolute Percentage Error: {mape:.1f}%")


----FOLD 1---
Epoch 1 | Loss: 0.019512367881324193
Epoch 100 | Loss: 0.001752262015201866
----FOLD 2---
Epoch 1 | Loss: 0.02052752295993779
Epoch 100 | Loss: 0.0015835673956532184
----FOLD 3---
Epoch 1 | Loss: 0.02128435787681031
Epoch 100 | Loss: 0.0013531490739383926
----FOLD 4---
Epoch 1 | Loss: 0.021452332379883282
Epoch 100 | Loss: 0.001529398846299681
----FOLD 5---
Epoch 1 | Loss: 0.02138558034210989
Epoch 100 | Loss: 0.0013216912401967668

Overall performance:
RMSE: 0.12
  Mean Absolute Error:  £14,165
  Mean Absolute Percentage Error: 8.5%
